# Face Filter App: Traditional Computer Vision

This notebook builds a face filter using **traditional** methods only

**Pipeline for each frame**

1. Capture a frame from the webcam (`cv2.VideoCapture`) or upload a photo
2. Convert to greyscale and equalise the histogram
3. Detect faces and eyes with a **Viola-Jones Haar cascade** (`cv2.CascadeClassifier`)
4. Apply the filter 
5. Display the result


**Requirements:** OpenCV Contrib 4.x, NumPy and Matplotlib. Run this notebook locally so it can reach your webcam. 
OpenCV Contrib is an extension of the OpenCV (Open Source Computer Vision) library in Python. It extends the capabilities of the OpenCV library in Python by offering additional features and modules. These extras aren't in the main OpenCV library for reasons like being experimental, serving niche needs, or relying on third-party tools. With these added modules, you can do a lot more in computer vision, like detecting features, tweaking images, tracking objects, diving into machine learning, and much more.


In [1]:
# Install if needed - OpenCV Contrib 4.x  this is a prerequisite for using Haarcascades
# !pip install "opencv-contrib-python<5" numpy matplotlib

In [1]:
# Shows where python is running from
import sys
print(sys.executable)   # the Python the kernel is actually running
print(sys.version)      # its Python version

C:\Users\CoreyGallagher-STUDE\Documents\cv_venv4\cv_env4\Scripts\python.exe
3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]


In [3]:
# Show a list of installed python libraries
# !pip list
# show the library details of opencv-contrib-python
# !pip show opencv-contrib-python

## Imports and loading the Haar cascades

In [2]:
import os
import time

import cv2
import numpy as np
import matplotlib.pyplot as plt

print("OpenCV version:", cv2.__version__)

CASCADE_DIR = "haarcascades/"   # local subfolder
if not os.path.isdir(CASCADE_DIR):
    CASCADE_DIR = cv2.data.haarcascades
print("Loading cascades from:", os.path.abspath(CASCADE_DIR))
face_cascade = cv2.CascadeClassifier(CASCADE_DIR+"haarcascade_frontalface_default.xml")
eye_cascade = cv2.CascadeClassifier(CASCADE_DIR+"haarcascade_eye.xml")

print("Face cascade loaded:", not face_cascade.empty())
print("Eye cascade loaded: ", not eye_cascade.empty())

OpenCV version: 4.14.0
Loading cascades from: C:\Users\CoreyGallagher-STUDE\Documents\cv_venv4\cv_env4\Lib\site-packages\cv2\data
Face cascade loaded: True
Eye cascade loaded:  True


## Part 1: Webcam video or upload a photo

Upload a Photo, or grab a single frame from the webcam or use the webcam as a video



### P1.1 Getting users choice

In [18]:
print("Filter System")
print("1. Upload Video/Photo")
print("2. Capture from webcam")
print("3. Live Webcam")

#Gets users choice 
while True:
    function_choice = input("Enter choice (1-3):")
    try:
        #tries to convert inout int integer
        function_choice = int(function_choice)
        #checks if it is within parameters
        if 1 <= function_choice <=3:
            break

        print("Invalid Input: must be between 1-3")
    #catches exception and displays error message when the inout is not an integer
    except ValueError:
        print("Invalid Input: must be between 1-3")

Filter System
1. Upload Video/Photo
2. Capture from webcam
3. Live Webcam


Enter choice (1-3): 1


### P1.2 Uploading Videos and Images

In [19]:
import tkinter as tk
from tkinter import filedialog

if function_choice == 1:
    root = tk.Tk()
    root.withdraw()

    # forces file selection window to top
    root.attributes('-topmost', True)
    root.update()

    # can only select certain file types e.g. mp4 and jpg.
    file_path = filedialog.askopenfilename(
        title="Select Image or Video",
            filetypes=[
            ("Media Files", "*.jpg *.jpeg *.png *.bmp *.mp4 *.avi *.mov"),
            ]
    )

    # closes window when finished
    root.destroy()
    #prints file path
    if file_path:
        print(f"Selected file: {file_path}")
    else:
        #prints when cancel or no file is opened
        print("No file selected.")



Selected file: C:/Users/CoreyGallagher-STUDE/OneDrive - Atlantic TU/Documents/Semester5_ 2026/Computer Vision/Python Labs/week4_manipulation/week4pb/SteveJobs.jpg


### P1.3 Webcam Capture

In [20]:
if function_choice == 2:
    # Connects to your computer's default camera
    cap = cv2.VideoCapture(0)
    
    # Get Height and Width from the feed.
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    writer = cv2.VideoWriter('webcam_capture.mp4', cv2.VideoWriter_fourcc(*'XVID'),25, (width, height))
    
    while True:
        ret, frame = cap.read()
        writer.write(frame)
        cv2.imshow('Saving this video to a file - Press ESC to exit',frame)
    
        # Press ESC to quit.
        if cv2.waitKey(1) & 0xFF == 27:
            break
    
    cap.release()
    writer.release()
    cv2.destroyAllWindows()

    print("Webcam Capture Saved")
    file_path = "webcam_capture.mp4"

## Part 3: Detect faces and eyes

Detect faces and eyes with a **Viola-Jones Haar cascade** (`cv2.CascadeClassifier`)

### P2.1 Differentiating videos from files

In [21]:
import os

#gets the extension of the selected file
extension = os.path.splitext(file_path)[1].lower()

#lists with image and video extensions
image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', 'webp']
video_extensions = ['.mp4', '.avi', '.mov', '.mkv', '.webm']

file_type = None

# checks if fiel extension matches any type and sets file type to it
if extension in image_extensions:
    file_type = "image"
elif extension in video_extensions:
    file_type = "video"
else:
    print("Invalid file type")

#### P2.2 Detecting faces

In [22]:
# Creates the Haar cascade for face detection
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

def detect_faces(frame):
    # sets the frame to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    #detects the coordinates of the faces
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(30, 30)
    )

    return faces

## Part 3: Apply the filter

### P3.1 Creating filter

In [37]:
# rectangle filter over face
def draw_rectangle(frame, faces):
    for (x, y, w, h) in faces:
        cv2.rectangle(
            frame,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )
 
    return frame

def blur_image(frame):
    
    # adds blur filter to image
    frame = cv2.GaussianBlur(frame,(5,5),0)
    
    return frame

### P3.2 Applying filter

In [39]:
if file_type == "image":
    #loads the selected image
    image = cv2.imread(file_path)

    #gets the coordinates of faces
    faces = detect_faces(image)

    #applies filter to faces
    image = draw_rectangle(image, faces)
    image = blur_image(image)
    
    cv2.imshow("Detected Faces", image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    
elif file_type == "video":
    #loads the video into opencv
    cap = cv2.VideoCapture(file_path)
    
    while True:
        
        #gets a frame of the video
        ret, frame = cap.read()
        
        if not ret:
            break

        #detects faces in frame
        faces = detect_faces(frame)

        #applies filter to frame
        frame = draw_rectangle(frame, faces)
        frame = blur_image(frame)
        
        #shows resutls
        cv2.imshow("Result", frame)
    
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()